# 🖐 Control de Volumen con Landmarks de Mano

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

Este laboratorio demuestra el uso de **MediaPipe Hand Landmarker** para detectar puntos clave de la mano en tiempo real y traducir esa información gestual en una acción concreta: **subir y bajar el volumen del sistema**.

---

## ✦ Concepto

La mano tiene **21 landmarks** numerados. Vamos a usar la **posición vertical (eje Y) de la muñeca** como control:

- Mano **arriba** → volumen **alto**
- Mano **abajo** → volumen **bajo**

```
  4
  |   8  12  16  20
  3   |   |   |   |
  |   7  11  15  19
  2   |   |   |   |
  |   6  10  14  18
  1   |   |   |   |
   \  5   9  13  17
    \ |   |   |   |
     [0: MUÑECA]      ← usamos este punto
```

El landmark `0` (muñeca) tiene coordenada `y` normalizada entre `0.0` (arriba de la imagen) y `1.0` (abajo). Invertimos esa escala para mapearla a volumen (0 %–100 %).

## Objetivo

El objetivo de este laboratorio es construir una aplicación de visión artificial en tiempo real que detecte la posición de la mano mediante **MediaPipe Hand Landmarker** y traduzca esa información gestual en una acción concreta del sistema operativo: el control del volumen de audio.

## Al terminar este material vas a poder:

1. Descargar e instanciar un modelo de detección de puntos clave (*landmarks*) de MediaPipe.
2. Implementar una función de mapeo entre una coordenada normalizada y un rango de valores de salida.
3. Desarrollar un loop de cámara en tiempo real con OpenCV.
4. Integrar la salida de un modelo de visión artificial con una acción del sistema operativo.

## Terminología clave (Microglosario)

*   **Landmark (Punto Clave):** Coordenada geométrica de referencia que el modelo aprende a localizar en la imagen. *Como las articulaciones y nudillos que reconocerías en la radiografía de una mano: hay 21 puntos definidos que el modelo aprende a ubicar con precisión en cada cuadro.*
*   **Coordenada normalizada:** Valor entre 0.0 y 1.0 que representa una posición relativa dentro de la imagen, independientemente de su resolución. *Como decir "el objeto está al 30 % del ancho de la pantalla" en lugar de "en el píxel 384": funciona igual en cualquier tamaño de ventana.*
*   **Suavizado exponencial:** Técnica que promedia gradualmente el valor nuevo con el anterior para eliminar saltos bruscos. *Como el comportamiento del termostato de un auto: no reacciona de golpe a cada cambio de temperatura, sino que avanza suavemente hacia el nuevo valor.*
*   **HUD (Heads-Up Display):** Superposición gráfica de información de estado sobre la imagen en tiempo real. *Como el velocímetro proyectado en el parabrisas de un avión: la información aparece encima de la escena sin interrumpirla.*
*   **Hand Landmarker:** Modelo preentrenado de MediaPipe que localiza los 21 puntos clave de la mano en cada cuadro de video. *Como un GPS que, en lugar de ciudades, mapea en tiempo real las articulaciones de los dedos.*

## Paso 1 — Preparar el entorno y el modelo

Instalá las dependencias desde la terminal con `uv sync` (o `uv add ...`) y luego ejecutá esta celda **una sola vez** para descargar el modelo pre-entrenado.

In [1]:
# Descarga el modelo Hand Landmarker de MediaPipe si no está disponible localmente.
# El modelo (.task) es un bundle TFLite que contiene la arquitectura y los pesos
# entrenados para detectar los 21 puntos clave de la mano.

import urllib.request, os

MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

# Elimina el archivo existente (aunque esté corrupto) y descarga de nuevo
if os.path.exists(MODEL_PATH):
    os.remove(MODEL_PATH)
    print("Archivo anterior eliminado.")

print("Descargando modelo...")
urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

tamaño_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f"✓ Descarga completa: {tamaño_mb:.2f} MB")



Archivo anterior eliminado.
Descargando modelo...
✓ Descarga completa: 7.46 MB


## Paso 2 — Funciones auxiliares

Definimos las funciones para:
1. Detectar la mano
2. Mapear la posición Y al rango de volumen
3. Ejecutar el cambio de volumen en el sistema operativo, incluido Windows con `pycaw`

In [2]:
# Configura el detector de manos de MediaPipe y define las funciones auxiliares de
# visualización y control de volumen multiplataforma (macOS, Linux, Windows).

import mediapipe as mp   # Framework de landmarks; proporciona el modelo Hand Landmarker
import cv2               # OpenCV para captura de video y dibujo sobre cuadros
import subprocess        # Necesario para lanzar comandos del SO que controlan el audio en macOS y Linux
import platform          # Permite detectar el SO en tiempo de ejecución para elegir la API de volumen correcta

# --- Configuración del Hand Landmarker ---
# Alias cortos para las clases de la Tasks API de MediaPipe (mejoran la legibilidad del código)
OpcionesBase          = mp.tasks.BaseOptions
DetectorManos         = mp.tasks.vision.HandLandmarker
OpcionesDetectorManos = mp.tasks.vision.HandLandmarkerOptions
ModoEjecucion         = mp.tasks.vision.RunningMode

opciones_landmarker = OpcionesDetectorManos(
    base_options=OpcionesBase(model_asset_path='hand_landmarker.task'),  # Ruta al modelo TFLite descargado
    running_mode=ModoEjecucion.IMAGE,   # Modo imagen (frame a frame); IMAGE no requiere timestamps como VIDEO
    num_hands=1,                        # Solo se analiza una mano para simplificar el control de volumen
    min_hand_detection_confidence=0.5,  # Umbral mínimo para iniciar el seguimiento de una mano nueva
    min_hand_presence_confidence=0.5,   # Umbral mínimo para mantener el seguimiento en frames siguientes
)


# --- Control de volumen según el sistema operativo ---

SISTEMA_OPERATIVO = platform.system()  # 'Darwin' = macOS, 'Linux', 'Windows'
print(f"✓ Sistema operativo detectado: {SISTEMA_OPERATIVO}")

VOLUMEN_WINDOWS = None               # Objeto COM de audio; se inicializa solo en Windows (None = sin soporte)
ADVERTENCIA_VOLUMEN_MOSTRADA = False  # Flag para mostrar el error de volumen una sola vez, no en cada frame

if SISTEMA_OPERATIVO == 'Windows':
    try:
        from comtypes import CoInitialize
        from pycaw.pycaw import AudioUtilities

        CoInitialize()  # Inicializa el subsistema COM de Windows, requerido por pycaw antes de cualquier llamada de audio
        dispositivos = AudioUtilities.GetSpeakers()    # Obtiene el dispositivo de salida de audio predeterminado
        VOLUMEN_WINDOWS = dispositivos.EndpointVolume  # Interfaz COM que permite leer y escribir el nivel de volumen
        print("✓ Control de volumen de Windows inicializado.")
    except Exception as e:
        VOLUMEN_WINDOWS = None
        print(f"Aviso: no se pudo inicializar pycaw ({e})")


def ajustar_volumen(nivel: int):
    """Establece el volumen del sistema operativo. nivel: 0-100"""
    global ADVERTENCIA_VOLUMEN_MOSTRADA
    nivel = max(0, min(100, nivel))  # Clampea el valor para evitar errores si el gesto excede el rango esperado
    try:
        if SISTEMA_OPERATIVO == 'Darwin':
            # osascript ejecuta AppleScript; es el único mecanismo nativo para controlar el audio en macOS
            subprocess.run(
                ['osascript', '-e', f'set volume output volume {nivel}'],
                capture_output=True  # Suprime la salida del subproceso para no contaminar la consola del notebook
            )
        elif SISTEMA_OPERATIVO == 'Linux':
            # amixer controla el mixer de ALSA (subsistema de audio estándar en Linux)
            subprocess.run(
                ['amixer', '-q', 'sset', 'Master', f'{nivel}%'],
                capture_output=True
            )
        elif SISTEMA_OPERATIVO == 'Windows':
            if VOLUMEN_WINDOWS is None:
                return
            VOLUMEN_WINDOWS.SetMute(0, None)  # Desmutea primero; si estaba muteado el cambio de nivel no tendría efecto
            VOLUMEN_WINDOWS.SetMasterVolumeLevelScalar(nivel / 100.0, None)  # La API COM espera un escalar 0.0–1.0
    except Exception as e:
        if not ADVERTENCIA_VOLUMEN_MOSTRADA:  # Evita repetir el aviso en cada frame del loop de video (hasta 30 fps)
            print(f"Aviso: no se pudo aplicar el volumen del sistema ({e})")
            ADVERTENCIA_VOLUMEN_MOSTRADA = True


# --- Mapeo de coordenada Y normalizada a porcentaje de volumen ---
# y=0.0 es la parte superior de la imagen → volumen alto
# y=1.0 es la parte inferior               → volumen bajo

def y_a_volumen(coordenada_y: float) -> int:
    """Convierte coordenada y normalizada [0,1] a porcentaje de volumen [0,100]."""
    # La inversión (1.0 - y) es necesaria porque en imagen el eje Y crece hacia abajo,
    # pero semánticamente "mano arriba" (y pequeño) debe significar "volumen alto"
    volumen = int((1.0 - coordenada_y) * 100)
    return max(0, min(100, volumen))


# --- Visualización: dibuja landmarks y HUD sobre el cuadro ---

COLOR_LANDMARK = (0, 255, 120)    # verde para los puntos clave
COLOR_CONEXION = (255, 255, 255)  # blanco para las líneas
COLOR_MUNECA   = (0, 120, 255)    # naranja para la muñeca (landmark 0), que es la referencia de posición para el volumen

# Pares de índices (a, b) que definen cada segmento a trazar.
# Los primeros 5 grupos son los 5 dedos (4 segmentos cada uno, de la base a la punta).
# Los últimos 3 pares son las conexiones transversales de la palma.
CONEXIONES_MANO = [
    (0,1),(1,2),(2,3),(3,4),         # Pulgar
    (0,5),(5,6),(6,7),(7,8),         # Índice
    (0,9),(9,10),(10,11),(11,12),    # Medio
    (0,13),(13,14),(14,15),(15,16),  # Anular
    (0,17),(17,18),(18,19),(19,20),  # Meñique
    (5,9),(9,13),(13,17)             # Arco de la palma
]


def dibujar_landmarks(cuadro, puntos_clave):
    alto, ancho = cuadro.shape[:2]

    # Los landmarks de MediaPipe son coordenadas normalizadas [0,1]; hay que escalarlas a píxeles
    puntos = []
    for lm in puntos_clave:
        x_pixel = int(lm.x * ancho)
        y_pixel = int(lm.y * alto)
        puntos.append((x_pixel, y_pixel))

    # Líneas de conexión entre los pares definidos en CONEXIONES_MANO
    for conexion in CONEXIONES_MANO:
        punto_inicio = conexion[0]
        punto_fin    = conexion[1]
        cv2.line(cuadro, puntos[punto_inicio], puntos[punto_fin], COLOR_CONEXION, 1, cv2.LINE_AA)

    # Círculos sobre cada landmark; la muñeca (i=0) se resalta porque es el punto de control del volumen
    for i in range(len(puntos)):
        px, py = puntos[i]
        color_punto = COLOR_MUNECA if i == 0 else COLOR_LANDMARK
        radio = 7 if i == 0 else 4
        cv2.circle(cuadro, (px, py), radio, color_punto, -1, cv2.LINE_AA)

    return puntos  # Devuelve posiciones en píxeles para que el llamador pueda usarlas (ej: calcular volumen)


def dibujar_hud(cuadro, volumen: int, mano_detectada: bool):
    alto, ancho = cuadro.shape[:2]

    # Barra de volumen en el lateral derecho
    barra_x, barra_y        = ancho - 50, 30
    barra_alto, barra_ancho = alto - 60, 30

    # Fondo gris de la barra
    cv2.rectangle(cuadro, (barra_x, barra_y), (barra_x + barra_ancho, barra_y + barra_alto), (60, 60, 60), -1)

    # Relleno proporcional al volumen; crece desde abajo hacia arriba para ser intuitivo
    relleno_alto  = int(barra_alto * volumen / 100)
    color_relleno = (0, 200 + int(55 * volumen / 100), 100)  # Verde más brillante cuanto mayor es el volumen (200→255)
    cv2.rectangle(cuadro,
                  (barra_x, barra_y + barra_alto - relleno_alto),  # Origen desde el fondo menos el relleno
                  (barra_x + barra_ancho, barra_y + barra_alto),
                  color_relleno, -1)

    # Borde de la barra
    cv2.rectangle(cuadro, (barra_x, barra_y), (barra_x + barra_ancho, barra_y + barra_alto), (180, 180, 180), 1)

    # Etiqueta con el porcentaje debajo de la barra y título encima
    cv2.putText(cuadro, f"{volumen}%", (barra_x - 5, barra_y + barra_alto + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(cuadro, "VOL", (barra_x + 2, barra_y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1, cv2.LINE_AA)

    # Estado de detección en la esquina superior izquierda; el color cambia para dar feedback visual inmediato
    estado       = "MANO DETECTADA" if mano_detectada else "Sin detección"
    color_estado = (0, 255, 120) if mano_detectada else (80, 80, 80)
    cv2.putText(cuadro, estado, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color_estado, 1, cv2.LINE_AA)

    # Instrucciones fijas en la parte inferior del cuadro
    cv2.putText(cuadro, "Mano arriba = volumen alto", (15, alto - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(cuadro, "Mano abajo  = volumen bajo", (15, alto - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(cuadro, "[Q] para salir", (15, alto - 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (120, 120, 120), 1, cv2.LINE_AA)


print("✓ Funciones auxiliares cargadas.")


✓ Sistema operativo detectado: Windows
✓ Control de volumen de Windows inicializado.
✓ Funciones auxiliares cargadas.


## Paso 3 — Loop principal

Ejecutá esta celda para iniciar la webcam. Una ventana de OpenCV se abrirá con la cámara en tiempo real.

- **Mové la mano** dentro del cuadro
- La barra lateral muestra el volumen en tiempo real
- **Presioná `Q`** para cerrar

> ℹ️ Si la cámara no abre o el volumen no cambia, cerrá otras apps que estén usando la webcam o el audio. En macOS, aceptá los permisos de cámara si el sistema los solicita.

In [6]:
# Loop principal de captura y procesamiento: lee cuadros de la cámara, detecta la mano,
# convierte la posición vertical de la muñeca en nivel de volumen y actualiza el audio del SO.

# ─── LOOP PRINCIPAL ────────────────────────────────────────────────────────────

captura = cv2.VideoCapture(0)   # 0 = cámara por defecto del sistema

# Suavizado exponencial del volumen para evitar saltos bruscos
SUAVIZADO         = 0.2   # entre 0 (muy lento) y 1 (sin suavizado)
volumen_suavizado = 50     # volumen inicial en porcentaje

# Limitamos la frecuencia de llamadas al SO para no sobrecargarlo
APLICAR_CADA_N_CUADROS = 5   # A 30 fps esto equivale a ~6 llamadas por segundo, suficiente para respuesta fluida
contador_cuadros        = 0
ultimo_volumen_aplicado = None  # Evita llamadas redundantes al SO cuando el volumen no cambió entre frames

print("✦ Cámara activa. Presioná Q en la ventana para cerrar.")

try:
    if not captura.isOpened():
        raise RuntimeError("No se pudo acceder a la cámara. Verificá los permisos.")

    with DetectorManos.create_from_options(opciones_landmarker) as landmarker:
        # El bloque 'with' garantiza que el modelo libere memoria y handles nativos al salir,
        # incluso si ocurre un error dentro del loop
        while True:
            lectura_exitosa, cuadro = captura.read()
            if not lectura_exitosa:  # La cámara se desconectó o se terminó el archivo de video
                break

            # Espejamos horizontalmente para una interacción más intuitiva
            cuadro = cv2.flip(cuadro, 1)  # flip=1: espejo horizontal; la mano derecha del usuario aparece a su derecha en pantalla

            # OpenCV captura en BGR; MediaPipe requiere RGB, por eso se convierte antes de enviar al modelo
            cuadro_rgb = cv2.cvtColor(cuadro, cv2.COLOR_BGR2RGB)
            imagen_mp  = mp.Image(image_format=mp.ImageFormat.SRGB, data=cuadro_rgb)  # Encapsula el array numpy en el formato de imagen que espera la Tasks API

            # Ejecutamos la detección de puntos clave
            deteccion = landmarker.detect(imagen_mp)

            mano_detectada   = len(deteccion.hand_landmarks) > 0
            volumen_objetivo = volumen_suavizado  # Si no hay mano, el objetivo se mantiene igual y el volumen no cambia

            if mano_detectada:
                puntos_clave      = deteccion.hand_landmarks[0]  # Primera mano detectada (configuramos num_hands=1)
                posicion_muneca_y = puntos_clave[0].y             # Landmark 0 = muñeca; es el punto de referencia para el control de volumen

                volumen_objetivo = y_a_volumen(posicion_muneca_y)  # Convierte la altura de la muñeca en porcentaje (muñeca arriba → volumen alto)
                dibujar_landmarks(cuadro, puntos_clave)

            # Suavizado exponencial (EMA): new = old + α * (target - old)
            # Cada frame avanza un 15 % de la diferencia restante, suavizando los cambios bruscos del gesto
            volumen_suavizado = volumen_suavizado + SUAVIZADO * (volumen_objetivo - volumen_suavizado)
            volumen_mostrado  = int(volumen_suavizado)

            # Aplicamos el volumen al SO cada N cuadros para no saturarlo
            contador_cuadros += 1
            if contador_cuadros % APLICAR_CADA_N_CUADROS == 0 and volumen_mostrado != ultimo_volumen_aplicado:
                ajustar_volumen(volumen_mostrado)
                ultimo_volumen_aplicado = volumen_mostrado  # Guarda el último valor enviado para evitar syscalls repetidas

            # Superponemos el HUD y mostramos el cuadro
            dibujar_hud(cuadro, volumen_mostrado, mano_detectada)
            cv2.imshow('Control de Volumen — MediaPipe Hands', cuadro)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                # waitKey(1): espera 1 ms y procesa eventos de la ventana; sin esta llamada la ventana se congela
                # & 0xFF: máscara para aislar el byte bajo del código de tecla (necesario en algunas plataformas)
                print("✦ Saliendo...")
                break

finally:
    captura.release()       # Libera la cámara para que otros procesos puedan acceder a ella
    cv2.destroyAllWindows() # Cierra todas las ventanas de OpenCV (no se cierran automáticamente)
    print("✓ Sesión cerrada.")


✦ Cámara activa. Presioná Q en la ventana para cerrar.
✦ Saliendo...
✓ Sesión cerrada.


---

## ✎ Para pensar antes de continuar

Antes de pasar a las variaciones, respondé estas preguntas mirando el código que acabás de ejecutar:

**1. ¿Qué pasaría si cambiás `SUAVIZADO` a `1.0`? ¿Y a `0.01`? ¿Qué trade-off controlás con ese parámetro?**

El parámetro SUAVIZADO (alpha del EMA)

La fórmula es: volumen_suavizado = volumen_suavizado + SUAVIZADO * (volumen_objetivo - volumen_suavizado)

SUAVIZADO = 1.0: el volumen salta directamente al valor objetivo en cada frame. Sin suavizado, cualquier temblor leve de la mano o detección ruidosa causa saltos bruscos de volumen.

SUAVIZADO = 0.01: el volumen avanza un 1% de la diferencia por frame. A 30 fps tardaría varios segundos en alcanzar el valor deseado; la respuesta sería tan lenta que parecería no funcionar.

Trade-off: velocidad de respuesta vs. estabilidad. Un alpha alto responde rápido pero es nervioso; uno bajo es estable pero lento. El 0.15 elegido es un punto intermedio que filtra el ruido del gesto sin introducir un retraso perceptible.

**2. ¿Por qué se invierte la coordenada Y en `y_a_volumen`? Describí con tus palabras cómo funciona el sistema de coordenadas de MediaPipe.**
La inversión de coordenada Y en y_a_volumen

MediaPipe usa un sistema de coordenadas donde el origen (0, 0) está en la esquina superior izquierda de la imagen, y el eje Y crece hacia abajo:

```text
(0.0, 0.0) ───────────▶ X
     │
     ├── mano arriba → y pequeño (~0.2)
     │
     ├── mano abajo  → y grande  (~0.8)
     │
     ▼
     Y
```


Sin la inversión, levantar la mano (y = 0.2) daría volumen 20 y bajarla (y = 0.8) daría volumen 80 — al revés de lo esperado. La operación (1.0 - coordenada_y) * 100 voltea esa lógica para que mano arriba = volumen alto.

**3. ¿Por qué el volumen se aplica cada `APLICAR_CADA_N_CUADROS` cuadros en lugar de en cada uno? ¿Qué consecuencias tendría hacerlo en cada frame?**
Por qué no aplicar el volumen en cada frame

Aplicar el volumen llama a una API del sistema operativo (osascript, amixer, o COM de Windows). Hacerlo en cada frame a 30 fps significa 30 llamadas por segundo al SO, lo que genera:

Overhead innecesario: la mayoría de esas llamadas enviarían el mismo valor (el volumen no cambia frame a frame de forma significativa).

Saturación de la API de audio: en Windows especialmente, las llamadas COM frecuentes pueden introducir latencia o inestabilidad.

CPU desperdiciada: cada llamada implica un cambio de contexto del proceso.

Con APLICAR_CADA_N_CUADROS = 5 se baja a ~6 llamadas/segundo, perceptualmente igual de fluido para el oído humano. La segunda condición volumen_mostrado != ultimo_volumen_aplicado agrega otra capa: si el valor no cambió, ni siquiera se hace la llamada.

---

## Para explorar en clase

Una vez que el ejemplo base funciona, estas variaciones son buenos ejercicios para extender el laboratorio:

### Variación 1 — Usar la distancia entre dedos
En lugar de la posición de la muñeca, usá la **distancia entre el pulgar (landmark 4) y el índice (landmark 8)** como control. Es el gesto clásico de "pinch-to-zoom".

```python
import math

def distancia(lm_a, lm_b):
    return math.sqrt((lm_a.x - lm_b.x)**2 + (lm_a.y - lm_b.y)**2)

d = distancia(puntos_clave[4], puntos_clave[8])  # pulgar - índice
# d ≈ 0.0 (dedos juntos) a 0.4 (dedos separados)
volumen_objetivo = int((d / 0.4) * 100)
```

### Variación 2 — Usar ambas manos
Cambiá `num_hands=2` en las opciones y usá una mano para el volumen y la otra para el brillo.

### Variación 3 — Contar dedos levantados
Detectá si cada dedo está extendido (comparando la punta con la articulación media) y mapeá el conteo (0–5) a valores discretos de volumen: 0 %, 20 %, 40 %, 60 %, 80 %, 100 %.

---

## Referencias

- [MediaPipe Hand Landmarker — Documentación oficial](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker)
- [Índice de landmarks de la mano](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker#hand_landmark_model_bundle)
- [pycaw — Control de volumen en Windows](https://github.com/AndreMiras/pycaw)